# Iowa Liquor Sales Analysis
Predict total_sales for store-category-month combinations

In [ ]:
# install packages
!pip install pandas scikit-learn numpy

In [ ]:
# load data
import pandas as pd
import numpy as np

df = pd.read_csv('../starter/data/raw/iowa_liquor_train.csv')
print(df.shape)
df.head()

In [ ]:
# check the data
print(df.info())
print(df.describe())

In [ ]:
# check missing values
print(df.isnull().sum())
df = df.dropna()
print("After dropping nulls:", df.shape)

In [ ]:
# feature engineering - category encoding
# take top 10 categories by frequency and make the rest "Other"
top_cats = df['category_name'].value_counts().head(10).index.tolist()
print("Top 10 categories:", top_cats)

df['category_encoded'] = df['category_name'].apply(lambda x: x if x in top_cats else 'Other')
print(df['category_encoded'].value_counts())

In [ ]:
# one hot encode categories
cat_dummies = pd.get_dummies(df['category_encoded'], prefix='cat')
df = pd.concat([df, cat_dummies], axis=1)
print("Shape after encoding:", df.shape)

In [ ]:
# seasonality features
import math

df['month_sin'] = df['month'].apply(lambda x: math.sin(2 * math.pi * x / 12))
df['month_cos'] = df['month'].apply(lambda x: math.cos(2 * math.pi * x / 12))
print("Added month_sin and month_cos")

In [ ]:
# log transform volume
df['log_volume'] = np.log1p(df['total_volume_liters'])
print("Added log_volume")
print("Min:", df['log_volume'].min(), "Max:", df['log_volume'].max())

In [ ]:
# prepare features and target
feature_cols = ['num_transactions', 'total_bottles', 'total_volume_liters',
                'avg_bottle_price', 'month_sin', 'month_cos', 'log_volume']

# add category dummy columns
cat_cols = [c for c in df.columns if c.startswith('cat_')]
feature_cols = feature_cols + cat_cols

print(f"Using {len(feature_cols)} features")
print(feature_cols)

X = df[feature_cols]
y = df['total_sales']

In [ ]:
# train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

In [ ]:
# train model
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)
print("Model trained")

In [ ]:
# predict
y_pred = model.predict(X_test)
print("Predictions made")
print("Sample predictions:", y_pred[:5])
print("Sample actuals:    ", y_test.values[:5])

In [ ]:
# evaluate - RMSE
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f"RMSE: {rmse:.4f}")

In [ ]:
# evaluate - R2
r2 = r2_score(y_test, y_pred)
print(f"R2: {r2:.4f}")

In [ ]:
# evaluate - MAE
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.4f}")

In [ ]:
# print all metrics again
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

rmse2 = mean_squared_error(y_test, y_pred, squared=False)
r2_2 = r2_score(y_test, y_pred)
mae2 = mean_absolute_error(y_test, y_pred)
print("=" * 40)
print("FINAL RESULTS")
print("=" * 40)
print(f"RMSE: {rmse2:.4f}")
print(f"R2:   {r2_2:.4f}")
print(f"MAE:  {mae2:.4f}")
print(f"Num features:      {len(feature_cols)}")
print(f"Train samples:     {X_train.shape[0]}")
print(f"Test samples:      {X_test.shape[0]}")
print("=" * 40)